# Phase 2 · 20단원 — 모델 모니터링과 드리프트

> 배포된 모델은 **에러 메시지 없이** 고장납니다.
> 점수는 계속 나오는데, 세상이 학습 데이터에서 멀어질 뿐이죠.
> 그 간격은 **감시해야만** 보입니다.

- 원문: <https://aiengineeringfromscratch.com/lesson?path=phases/02-ml-fundamentals/20-model-monitoring-and-drift>
- 예상 시간: 110분
- 먼저 보면 좋은 것: 9단원(모델 평가), 16단원(이상 탐지), 19단원(시스템 설계)
- 표시 규칙: 📗 는 원문에 있는 내용, ✨ 는 제가 넣은 예시·실험, ⚠️ 는 원문과 측정이 달랐던 곳입니다.

---

## 이 단원이 푸는 문제

결제 위험 모델이 **정확도 95%** 로 출시됩니다.

**두 달 뒤** — 마케팅 캠페인으로 결제 금액이 큰 앱 사용자가 몰려옵니다.
**넉 달 뒤** — 사기꾼들이 수법을 바꿔서, 예전엔 안전했던 결제가 차지백으로 돌아오기 시작합니다.

이 모든 기간 동안 모델 서비스는 **모든 요청에 HTTP 200 을 돌려줍니다.**
지연도 정상, 로그에도 에러 하나 없습니다.

처음 이상을 알아챈 건 **몇 주 뒤에 올라온 재무팀의 차지백 보고서**였습니다.

> **소프트웨어 모니터링**은 서비스가 **응답하는지** 봅니다.
> **모델 모니터링**은 그 응답이 **아직도 맞는지**, 그리고
> 들어오는 데이터가 **아직도 학습 데이터를 닮았는지** 봅니다.

이 단원은 그 두 번째 일을 하는 **통계·창·경보 정책**을 만듭니다.

## 그래서 이 단원이 만드는 것

전부 표준 라이브러리로 직접 만듭니다. 📗 원문 코드도 그렇고요.

| 도구 | 하는 일 |
| --- | --- |
| **PSI** | 수치형 특징의 분포가 얼마나 움직였나 |
| **KS 통계량** | 구간을 안 나누고 분포 차이를 잼 |
| **카이제곱 + Cramér's V** | 범주형 특징(결제 채널 등)의 변화 |
| **슬라이딩 창** | 기준 구간과 최근 구간을 비교 |
| **경보 정책** | 거짓 경보를 줄여 사람이 알람을 계속 읽게 만듦 |

그리고 **드리프트를 두 번 주입한 9000건짜리 흐름**에 전부 돌려서,
어느 감시기가 어느 변화를 잡는지 봅니다.

---

## 이걸 배우면 뭘 할 수 있나

> **"모델 배포했는데 잘 돌아가는지 어떻게 알죠?"**

이 질문에 **감시할 네 가지**와 **각각이 못 보는 것**을 짚어 줄 수 있게 됩니다.

| 질문 | 어디서 | 이 단원의 답 |
| --- | --- | --- |
| 뭘 감시하죠? | 1절 | 시스템 / 입력 / 예측 / 결과. 앞 셋은 빠르고 마지막 하나만 정답을 압니다 |
| 분포가 변한 걸 어떻게 재죠? | 4~8절 | PSI · KS · 카이제곱을 직접 만듭니다 |
| 경보를 하루 몇 번 울려야 하죠? | 9절 | 보정 · 효과 크기 · 지속 · 등급 분리 |
| 드리프트 났는데 재학습하면 되나요? | 12절 | **종류에 따라 다릅니다.** 재학습이 소용없는 경우가 있습니다 |

그리고 가장 중요한 것 하나. **입력 감시로는 절대 볼 수 없는 고장**이 있습니다(11절).
이걸 모르면 "입력 지표 전부 정상입니다" 라고 보고하면서 모델이 망가지는 걸 놓칩니다.

---

## 이 단원의 줄거리

19단원에서 모델을 **설계하고 배포**했습니다. 20단원은 **배포한 다음**입니다.

**1절에서 감시할 것을 네 층으로 나눕니다.**
시스템 · 입력 · 예측 · 결과. 앞 셋은 **즉시** 볼 수 있고, 마지막 하나만 **정답을 압니다.**
그런데 그게 제일 늦게 옵니다. 이 비대칭이 단원 전체를 끌고 갑니다.

**2절은 드리프트가 왜 한 종류가 아닌지**입니다.
입력이 변한 건지, 정답 비율이 변한 건지, **같은 입력의 정답이 변한 건지**가 전부 다릅니다.
그리고 어느 감시기가 잡는지도 다릅니다.

**3절은 레이블이 늦게 오는 문제**입니다. 사기 차지백은 몇 주 뒤에 옵니다.
그동안 우리에겐 입력과 예측뿐이죠.

**4절에서 창을 정하고, 5~8절에서 통계를 직접 만듭니다.**
PSI, KS, 카이제곱 셋입니다. 각각 하나씩 ⚠️ 함정이 있습니다.

- PSI — 흔히 쓰는 **0.1 / 0.25 기준선**이 절대 기준이 아닙니다(6절)
- KS — 표본이 커지면 **p값이 거짓말**을 합니다(8절)

**9절이 실무에서 가장 중요합니다.** 거짓 경보를 줄이는 네 규칙이요.
50개 특징을 매시간 검사하면 **아무 일 없어도 하루 60번 경보가 울립니다.**

**10절에서 전부 합쳐 9000건짜리 흐름에 돌립니다.**
드리프트를 두 번 주입해 뒀습니다. 어느 감시기가 어느 걸 잡는지 보세요.

**11절이 이 단원의 핵심입니다.** 두 번째 드리프트는 **입력 감시기가 하나도 못 잡습니다.**
10개 창 전부 조용합니다. 그런데 정확도는 0.946에서 0.760으로 떨어집니다.

**12절**에서 드리프트 종류별로 **뭘 해야 하는지** 정리합니다.
재학습이 답이 아닌 경우가 있습니다.

> 한 줄로 줄이면 이렇습니다.
> **입력 감시는 빠르지만 눈이 멀었고, 결과 감시는 정확하지만 늦습니다.
> 그래서 둘 다 필요하고, 경보 정책이 둘을 쓸 만하게 만듭니다.**

<details>
<summary>절 목록 (펼쳐 보기)</summary>

| 절 | 내용 | 출처 |
| --- | --- | --- |
| 1 | 무엇을 감시하나 — 네 개의 층 | 📗 원문 |
| 2 | 드리프트 세 종류 | 📗 원문 |
| 3 | 레이블은 늦게 옵니다 | 📗 원문 |
| 4 | 기준 창과 현재 창 | 📗 원문 |
| 5 | PSI 직접 만들기 | 📗 원문 |
| 6 | ⚠️ PSI 의 0.1 / 0.25 는 절대 기준이 아닙니다 | 📗 원문 + ✨ 측정 |
| 7 | KS 통계량 — 구간 없이 재기 | 📗 원문 |
| 8 | ⚠️ 표본이 커지면 p값이 거짓말합니다 | 📗 원문 + ✨ 측정 |
| 9 | 카이제곱과 경보 정책 | 📗 원문 |
| 10 | 드리프트를 주입한 흐름에 전부 돌리기 | 📗 원문 |
| 11 | ⚠️ 입력 감시의 사각지대 + 창 크기 | 📗 원문 + ✨ 측정 |
| 12 | 재학습 방아쇠 — 종류별로 할 일 | 📗 원문 |
| 13 | 정리 · Ship It · 용어 · 퀴즈 | 📗 원문 |

</details>

---

## 0. 준비

In [1]:
import bisect
import math
import random
import warnings
from collections import deque
from dataclasses import dataclass, field

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.font_manager import findfont, FontProperties

warnings.filterwarnings("ignore")

후보들 = ["AppleGothic", "Apple SD Gothic Neo", "Malgun Gothic", "NanumGothic"]
찾은것 = None
for 후보 in 후보들:
    try:
        if 후보.lower().replace(" ", "") in findfont(FontProperties(family=후보)).lower().replace(" ", ""):
            찾은것 = 후보
            break
    except Exception:
        pass
matplotlib.rcParams["font.family"] = ["DejaVu Sans"] + ([찾은것] if 찾은것 else [])
matplotlib.rcParams["axes.unicode_minus"] = False
matplotlib.rcParams["figure.dpi"] = 110

print("준비 끝!  한글 폰트:", 찾은것 or "(없음)")

준비 끝!  한글 폰트: AppleGothic


---

## 1. 무엇을 감시하나 — 네 개의 층

<sub>📗 원문 "What to monitor"</sub>

📗 원문은 네 층을 감시하라고 합니다. **앞 셋은 조기 경보**를 주고,
**마지막 하나만 맞는지를 잽니다.** 그런데 그게 늦게 옵니다.

| 층 | 신호 | 언제 보나 | 답하는 질문 |
| --- | --- | --- | --- |
| **시스템 상태** | p50·p99 지연, 에러율, 처리량, 메모리 | 즉시 | 서비스가 응답하나? |
| **입력** | 특징별 분포, 결측률, 스키마, 새 범주 | 즉시 | 들어오는 데이터가 학습 데이터를 닮았나? |
| **예측** | 점수 분포, 양성 비율, 기본값 비율 | 즉시 | 모델이 전처럼 행동하나? |
| **결과** | 정확도, 정밀도, 보정, 비즈니스 지표 | **레이블 지연 후** | 모델이 아직 맞나? |

### 🔑 이 표의 비대칭이 단원 전체를 설명합니다

**맨 위 세 줄은 "즉시"**, 맨 아래 한 줄만 "나중에" 입니다.
그런데 **정답을 아는 건 맨 아래 한 줄뿐**이에요.

> 📗 원문: 이 단원의 주 소재는 **입력 감시**입니다. 레이블이 오기 전에 작동하니까요.
> 그런데 한계가 있고, 데모가 그걸 보여 줍니다.
> **어떤 변화는 입력을 전혀 건드리지 않습니다.**

마지막 문장을 기억해 두세요. **11절이 그 이야기입니다.**

> **🧩 이 절의 결론**
>
> 감시는 네 층입니다. **앞 셋은 빠르고 마지막 하나만 정답을 압니다.**
> 이 비대칭 때문에 둘 다 필요합니다. 빠른 것만 보면 놓치고, 정확한 것만 보면 늦습니다.
>
> **➡️ 다음**
>
> 그런데 '데이터가 변했다'는 말이 **한 가지 뜻이 아닙니다.**
> 세 종류로 나뉘고, 각각 잡히는 감시기가 다릅니다. 2절입니다.

---

## 2. 드리프트 세 종류

<sub>📗 원문 "Three kinds of drift"</sub>

입력 `x` 와 정답 `y` 의 결합 분포를 **두 가지 방법으로** 쪼갤 수 있습니다.

$$P(x, y) = P(x) \cdot P(y \mid x) = P(y) \cdot P(x \mid y)$$

**드리프트 종류마다 쪼갠 조각 중 다른 게 변합니다.**

| 드리프트 | 변하는 것 | 그대로인 것 | 결제 예시 | 어느 감시기가 보나 |
| --- | --- | --- | --- | --- |
| **공변량 드리프트** | $P(x)$ | $P(y \mid x)$ | 결제액 큰 앱 사용자가 늘어남 | **입력 감시** |
| **사전확률 드리프트** | $P(y)$ | $P(x \mid y)$ | 공격 기간에 사기율이 두 배. 사기 모습은 그대로 | 예측·결과 감시 |
| **개념 드리프트** | $P(y \mid x)$ | 그대로일 수 있음 | **같은 결제액인데 위험도가 달라짐** | **결과 감시만** |

### 🔑 맨 아랫줄의 "결과 감시만" 이 문제입니다

개념 드리프트는 **입력이 하나도 안 변해도** 생길 수 있습니다.
똑같은 10만 원짜리 결제가, 어제는 안전했는데 오늘은 사기인 거죠.

그러면 입력 감시기는 **아무것도 못 봅니다.** 입력이 안 변했으니까요.
정답이 올 때까지 기다려야 합니다. 그게 3절의 레이블 지연 문제고요.

> 📗 Gama 등은 $P(y \mid x)$ 가 변하는 걸 **진짜 개념 드리프트**,
> $P(y \mid x)$ 는 그대로인 채 $P(x)$ 만 변하는 걸 **가상 드리프트**라고 부릅니다.
> 같은 설문은 드리프트를 시간 양상으로도 나눕니다 — 갑작스러운 / 점진적 / 서서히 / 되돌아오는.

### 📗 공변량 드리프트가 늘 해로운 건 아닙니다

모델이 $P(y \mid x)$ 를 잘 배웠다면, 입력이 어디로 움직여도 **각 입력에 대해선 여전히 맞습니다.**

다만 트래픽이 **학습 데이터가 거의 없던 영역**이나
**결정 경계 근처**로 옮겨 가면 정확도가 떨어질 수 있습니다.

> **🧩 이 절의 결론**
>
> 드리프트는 셋입니다. **입력이 변한 것 / 정답 비율이 변한 것 / 같은 입력의 정답이 변한 것.**
> 마지막 하나(개념 드리프트)는 **입력이 그대로일 수 있어서** 입력 감시가 못 봅니다.
>
> **➡️ 다음**
>
> 그럼 결과 감시를 보면 되겠네요. 그런데 **결과는 늦게 옵니다.**
> 얼마나 늦는지가 3절입니다.

---

## 3. 레이블은 늦게 옵니다

<sub>📗 원문 "Delayed labels"</sub>

예측을 한 다음에 **정답이 옵니다.** 얼마나 늦는지는 문제마다 다릅니다.

| 문제 | 정답이 오는 시점 |
| --- | --- |
| 광고 클릭 | 몇 초 |
| 30일 내 이탈 | 30일 |
| 카드 사기 | **차지백이 들어올 때. 보통 몇 주 뒤** |
| 대출 연체 | 몇 달에서 몇 년 |

그 기간 동안 우리에겐 **입력과 예측만** 있고 **결과는 없습니다.**

> 📗 원문이 말하는 대처:
> 입력과 예측을 감시하고, 레이블이 도착하면 붙이고,
> **레이블이 붙은 부분에 대해서만** 결과 지표를 계산합니다.

이 노트북의 데모는 **레이블 지연을 500건**으로 둡니다.
그래서 결과 감시기는 **늘 500건 과거를 보고 있습니다.**

### 🔑 이게 11절에서 숫자로 나타납니다

개념 드리프트를 6500번째 이벤트에 주입하는데, 정확도 경보는 **7500에 울립니다.**
1000건 늦은 거죠. 그중 **500건이 레이블 지연** 때문입니다.
나머지 500건은 창 크기 때문이고요. 이것도 11절에서 측정합니다.

> **🧩 이 절의 결론**
>
> 결과 지표는 **레이블이 온 만큼만** 계산할 수 있습니다.
> 데모는 500건 지연이고, 그래서 결과 감시기는 늘 500건 과거를 봅니다.
>
> **➡️ 다음**
>
> 이제 재는 도구를 만듭니다. 그 전에 **무엇과 무엇을 비교할지** 정해야죠. 4절입니다.

---

## 4. 기준 창과 현재 창

<sub>📗 원문 "Reference and current windows"</sub>

모든 드리프트 검정은 **표본 두 개를 비교**합니다.

- **기준 창** — 학습 데이터, 또는 손으로 확인해 둔 운영 기간입니다.
  **재학습할 때까지 고정**입니다.
- **현재 창** — 최근 이벤트들. 마지막 `N` 건이거나 마지막 `T` 시간이거나.

**슬라이딩 창**은 창 크기보다 작은 폭으로 움직여서 **연속된 창이 겹칩니다.**
**텀블링 창**은 창 크기만큼 통째로 움직여서 안 겹치고요.

### 📗 창 크기는 맞교환입니다

| 창 크기 | 장점 | 단점 |
| --- | --- | --- |
| **작게** | 빨리 반응합니다 | **잡음을 드리프트로 오인**합니다 |
| **크게** | 안정적입니다 | **늦게 보고**합니다 |

이 맞교환이 교과서에 늘 나오는 설명인데, **11절에서 직접 재 봅니다.**
측정해 보면 생각보다 단순하지 않습니다.

### 📗 계절성이 창을 망칩니다

월요일 트래픽과 토요일 트래픽은 많은 제품에서 다릅니다.
**기준 창이 한 주기를 다 덮게** 하거나,
**이전 주기의 같은 시점**과 비교하세요.

> **🧩 이 절의 결론**
>
> 비교는 늘 **고정된 기준 창**과 **움직이는 현재 창** 사이에서 일어납니다.
> 창 크기는 **반응 속도 ↔ 잡음**의 맞교환이라고들 합니다. 11절에서 재 봅니다.
>
> **➡️ 다음**
>
> 비교할 두 표본이 생겼습니다. 이제 **얼마나 다른지 재는 법**입니다.
> 첫 번째 도구는 신용평가에서 온 PSI 입니다. 5절입니다.

---

## 5. PSI 직접 만들기

<sub>📗 원문 "Population Stability Index"</sub>

**PSI**(Population Stability Index)는 신용평가에서 왔습니다.

### 만드는 순서

1. **기준 값들을 구간으로 자릅니다.** 보통 10개 구간인데,
   **기준 데이터의 십분위수**에서 자릅니다. 그러면 각 구간에 기준의 10%씩 들어가죠
2. **현재 값들이 각 구간에 몇 %씩 들어가는지** 셉니다
3. 구간마다 항을 하나씩 만들어 **더합니다**

$$\text{PSI} = \sum_{\text{구간}} (\text{현재비율} - \text{기준비율}) \cdot \ln\left(\frac{\text{현재비율}}{\text{기준비율}}\right)$$

### 🔑 두 가지 주의

**① 비율이 0 이면 로그가 무한대가 됩니다.**
그래서 모든 비율을 **0.0001 같은 작은 값으로 하한을 둡니다.**

**② PSI 는 대칭입니다.**
현재→기준 KL 발산과 기준→현재 KL 발산을 **더한 값**과 같습니다.
어느 쪽을 기준으로 두든 같은 값이 나온다는 뜻이에요.

### 📗 실무에서 쓰는 해석 밴드

| PSI | 해석 |
| --- | --- |
| 0.1 미만 | 안정 |
| 0.1 ~ 0.25 | 중간 정도 이동 |
| 0.25 초과 | 유의미한 이동 |

> 📗 원문이 바로 경고합니다: **이건 관행이지 검정이 아닙니다.**
> PSI 는 **구간 수가 늘면 커지고 표본이 커지면 작아집니다.**

이 경고가 얼마나 중요한지 **6절에서 숫자로** 봅니다.

> **실험 5** · PSI 를 만들고 정상 범위를 확인합니다
>
> **🎯 배울 것** — 드리프트가 **없을 때** PSI 가 얼마나 나오는지 알아야 기준선을 정할 수 있습니다.
>
> **🔬 하는 일** — PSI 를 밑바닥부터 만들고, 같은 분포에서 뽑은 창 40개에 돌려 봅니다. 📗 원문은 0.01~0.04 라고 합니다.

In [2]:
# 📗 원문의 quantile_edges / bin_fractions / psi 를 옮깁니다
def 십분위경계(기준값들, 구간수=10):
    정렬 = sorted(기준값들)
    n = len(정렬)
    경계 = [정렬[min(n - 1, (n * i) // 구간수)] for i in range(1, 구간수)]
    return sorted(set(경계))


def 구간비율(값들, 경계):
    개수 = [0] * (len(경계) + 1)
    for v in 값들:
        개수[bisect.bisect_right(경계, v)] += 1
    return [c / len(값들) for c in 개수]


def PSI계산(기준비율, 현재비율, eps=1e-4):
    합 = 0.0
    for 기, 현 in zip(기준비율, 현재비율):
        기, 현 = max(기, eps), max(현, eps)
        합 += (현 - 기) * math.log(현 / 기)
    return 합


def PSI(기준값들, 현재값들, 구간수=10):
    경계 = 십분위경계(기준값들, 구간수)
    return PSI계산(구간비율(기준값들, 경계), 구간비율(현재값들, 경계))


rng = random.Random(0)
기준 = [rng.gauss(100, 20) for _ in range(2000)]       # 학습 데이터라고 칩시다

# 드리프트가 전혀 없는 창 40개
깨끗 = []
for s in range(40):
    r = random.Random(100 + s)
    깨끗.append(PSI(기준, [r.gauss(100, 20) for _ in range(500)]))

# 평균이 100 -> 118 로 이동한 창
이동 = []
for s in range(40):
    r = random.Random(100 + s)
    이동.append(PSI(기준, [r.gauss(118, 24) for _ in range(500)]))

print("기준 창: 평균 100, 표준편차 20 짜리 2000건 (학습 데이터)")
print("현재 창: 500건씩 40개\n")
print(f"{'현재 창의 분포':<30}{'최소':>8}{'중앙':>8}{'최대':>8}")
print("-" * 56)
print(f"{'똑같음 (평균 100, 표준편차 20)':<28}{min(깨끗):>8.3f}{sorted(깨끗)[20]:>8.3f}{max(깨끗):>8.3f}")
print(f"{'이동함 (평균 118, 표준편차 24)':<28}{min(이동):>8.3f}{sorted(이동)[20]:>8.3f}{max(이동):>8.3f}")

print(f"\n해석 밴드:  0.1 미만 안정  |  0.1~0.25 중간  |  0.25 초과 유의미")
print(f"\n=> 드리프트가 없을 때 PSI 는 {min(깨끗):.3f} ~ {max(깨끗):.3f} 사이입니다.")
print(f"   📗 원문도 '0.01 에서 0.04 사이' 라고 합니다. ✔ 맞습니다.")
print(f"   40개 창 중 0.1 을 넘은 건 {sum(1 for v in 깨끗 if v >= 0.1)}개입니다.")
print(f"\n   평균이 18 움직이니 중앙값이 {sorted(이동)[20]:.3f} 로 뜁니다. 밴드로 치면 '유의미한 이동'이죠.")
print(f"   차이가 {sorted(이동)[20] / sorted(깨끗)[20]:.0f}배쯤 납니다. 구별이 쉽습니다.")

기준 창: 평균 100, 표준편차 20 짜리 2000건 (학습 데이터)
현재 창: 500건씩 40개

현재 창의 분포                            최소      중앙      최대
--------------------------------------------------------
똑같음 (평균 100, 표준편차 20)          0.009   0.024   0.046
이동함 (평균 118, 표준편차 24)          0.435   0.653   0.774

해석 밴드:  0.1 미만 안정  |  0.1~0.25 중간  |  0.25 초과 유의미

=> 드리프트가 없을 때 PSI 는 0.009 ~ 0.046 사이입니다.
   📗 원문도 '0.01 에서 0.04 사이' 라고 합니다. ✔ 맞습니다.
   40개 창 중 0.1 을 넘은 건 0개입니다.

   평균이 18 움직이니 중앙값이 0.653 로 뜁니다. 밴드로 치면 '유의미한 이동'이죠.
   차이가 27배쯤 납니다. 구별이 쉽습니다.


> **👀 주목할 곳** — 두 줄의 `최대` 와 `최소` 를 보세요. 겹치는 구간이 전혀 없습니다.
>
> **✅ 알게 된 사실** — 드리프트가 없을 때 PSI 는 0.01~0.05 에 머뭅니다. 📗 원문의 '0.01~0.04' 와 맞습니다.
평균이 18 움직이면 0.6 근처로 뜁니다. **20배 넘게** 차이가 나서 구별이 쉽습니다.

> **🧩 이 절의 결론**
>
> PSI 는 **구간별 비율 차이에 로그 비를 곱해 더한** 값입니다.
> 드리프트가 없으면 0.05 아래, 뚜렷한 이동이면 0.5 이상으로 확실히 갈립니다.
>
> **➡️ 다음**
>
> 그런데 방금 '0.1 미만이면 안정' 이라는 밴드를 썼죠.
> ⚠️ **그 밴드가 늘 맞는 게 아닙니다.** 6절에서 깨 봅니다.

---

## 6. ⚠️ PSI 의 0.1 / 0.25 는 절대 기준이 아닙니다

<sub>📗 원문 "Treat these bands as a convention, not a test" + ✨ 측정</sub>

📗 원문은 한 문장으로 경고하고 넘어갑니다.

> PSI grows with the number of bins and shrinks with sample size.
> (PSI 는 **구간 수가 늘면 커지고 표본이 커지면 작아집니다.**)

**얼마나 커지고 작아지는지**를 재 보면 꽤 놀랍습니다.
**드리프트가 하나도 없는데도** 밴드를 넘길 수 있습니다.

> **실험 6** · ⚠️ 드리프트 없이 '중간 이동' 판정 만들기
>
> **🎯 배울 것** — PSI 숫자 하나만 보고 판단하면 안 되는 이유를 **반례로** 확인합니다.
>
> **🔬 하는 일** — 똑같은 분포에서 뽑은 데이터로, ① 구간 수를 5→50 으로 ② 창 크기를 100→2000 으로 바꿔 가며 PSI 를 잽니다.

In [3]:
# ✨ 추가 — 드리프트가 0 인 상태에서 구간 수와 창 크기만 바꿉니다
print("① 구간 수만 바꿉니다 (창 500건, 드리프트 없음)\n")
print(f"{'구간 수':>8}{'PSI 중앙값':>12}{'0.1 넘은 창':>14}{'판정':>14}")
print("-" * 52)
for 구간수 in (5, 10, 20, 50):
    v = []
    for s in range(40):
        r = random.Random(200 + s)
        v.append(PSI(기준, [r.gauss(100, 20) for _ in range(500)], 구간수))
    중앙 = sorted(v)[20]
    판정 = "안정" if 중앙 < 0.1 else ("중간 이동" if 중앙 < 0.25 else "유의미")
    print(f"{구간수:>8}{중앙:>12.3f}{sum(1 for x in v if x >= 0.1):>12}/40{판정:>14}")

print("\n\n② 창 크기만 바꿉니다 (구간 10개, 드리프트 없음)\n")
print(f"{'창 크기':>8}{'PSI 중앙값':>12}{'PSI 최대':>11}{'0.1 넘은 창':>14}{'판정':>14}")
print("-" * 62)
for 창 in (100, 200, 500, 1000, 2000):
    v = []
    for s in range(40):
        r = random.Random(300 + s)
        v.append(PSI(기준, [r.gauss(100, 20) for _ in range(창)]))
    중앙 = sorted(v)[20]
    판정 = "안정" if 중앙 < 0.1 else ("중간 이동" if 중앙 < 0.25 else "유의미")
    print(f"{창:>8}{중앙:>12.3f}{max(v):>11.3f}{sum(1 for x in v if x >= 0.1):>12}/40{판정:>14}")

print(f"\n=> ⚠️ 두 표 모두 드리프트가 0 입니다. 똑같은 분포에서 뽑았어요.")
print(f"\n   ① 구간을 50개로 쪼개면 중앙값이 0.132, 40개 창 중 38개가 0.1 을 넘습니다.")
print(f"      구간이 잘게 쪼개질수록 각 구간의 기대 건수가 줄어서 비율이 들쭉날쭉해지거든요.")
print(f"\n   ② 창을 100건으로 줄이면 중앙값이 0.111, 40개 창 중 23개가 0.1 을 넘습니다.")
print(f"      표본이 작으면 비율 추정이 부정확해서 그렇습니다.")
print(f"\n   📝 그래서 0.1 / 0.25 밴드는 구간 10개 + 넉넉한 창 을 전제로 한 관행입니다.")
print(f"      설정을 바꾸면 밴드도 다시 정해야 합니다.")
print(f"      정하는 법은 간단합니다. 5절처럼 드리프트 없는 기간에 돌려 보고,")
print(f"      거기서 나온 최댓값 위로 기준선을 긋는 겁니다.")

① 구간 수만 바꿉니다 (창 500건, 드리프트 없음)

    구간 수     PSI 중앙값      0.1 넘은 창            판정
----------------------------------------------------
       5       0.010           0/40            안정
      10       0.021           0/40            안정
      20       0.058           0/40            안정
      50       0.132          38/40         중간 이동


② 창 크기만 바꿉니다 (구간 10개, 드리프트 없음)

    창 크기     PSI 중앙값     PSI 최대      0.1 넘은 창            판정
--------------------------------------------------------------
     100       0.111      0.254          23/40         중간 이동
     200       0.046      0.117           3/40            안정
     500       0.022      0.053           0/40            안정
    1000       0.012      0.036           0/40            안정
    2000       0.008      0.020           0/40            안정

=> ⚠️ 두 표 모두 드리프트가 0 입니다. 똑같은 분포에서 뽑았어요.

   ① 구간을 50개로 쪼개면 중앙값이 0.132, 40개 창 중 38개가 0.1 을 넘습니다.
      구간이 잘게 쪼개질수록 각 구간의 기대 건수가 줄어서 비율이 들쭉날쭉해지거든요.

   ② 창을 100건으로 줄이면 중앙값이 0.111, 40개 창 중 23개가 0.1 을 넘습니다

> **👀 주목할 곳** — 두 표의 `판정` 열. **드리프트가 0 인데** '중간 이동'이 나옵니다.
>
> **✅ 알게 된 사실** — ⚠️ 구간 50개면 PSI 중앙값이 0.132, 창 100건이면 0.111 입니다. 둘 다 '안정' 밴드를 넘습니다.
**0.1 / 0.25 는 구간 10개 + 넉넉한 창을 전제로 한 관행**입니다.
설정이 다르면 드리프트 없는 기간에 직접 돌려 보고 기준선을 다시 그어야 합니다.

> **🧩 이 절의 결론**
>
> ⚠️ PSI 밴드는 **설정에 딸린 값**입니다. 구간 수와 창 크기가 바뀌면 같이 바뀝니다.
> 남이 쓰는 0.1 을 그대로 가져오면 안 되고, **내 데이터로 정상 범위를 재야** 합니다.
>
> **➡️ 다음**
>
> PSI 는 구간을 나눠야 해서 이 문제가 생겼습니다.
> **구간을 안 나누는** 방법도 있습니다. 7절의 KS 통계량입니다.

---

## 7. KS 통계량 — 구간 없이 재기

<sub>📗 원문 "Kolmogorov-Smirnov statistic"</sub>

**두 표본 KS 통계량 `D`** 는 두 **누적분포함수** 사이의 **가장 큰 세로 간격**입니다.

$$D = \max_x \left\lvert F_{\text{기준}}(x) - F_{\text{현재}}(x) \right\rvert$$

| D 값 | 뜻 |
| --- | --- |
| 0 | 두 표본이 똑같음 |
| 1 | 두 표본이 전혀 안 겹침 |

**구간을 나눌 필요가 없습니다.** PSI 의 6절 문제 중 절반이 사라지는 거죠.

### 🔑 누적분포함수가 뭔가요

값을 작은 것부터 세워 놓고 **"이 값 이하가 전체의 몇 %인가"** 를 그린 겁니다.
항상 0 에서 시작해 1 로 끝나는 **계단 모양**이고요.

두 표본의 계단을 겹쳐 그렸을 때 **가장 많이 벌어진 지점의 간격**이 `D` 입니다.

### 📗 p값은 어디서 오나

콜모고로프 분포에서 나옵니다. **유효 표본 크기**는 $n \cdot m / (n + m)$ 이고요.

$$p = 2\sum_{k=1}^{\infty} (-1)^{k-1} e^{-2k^2\lambda^2}, \qquad \lambda = \sqrt{\frac{nm}{n+m}} \cdot D$$

구현은 아래 코드에 있습니다. 급수를 100항쯤 더하면 충분히 수렴합니다.

> **실험 7** · KS 통계량 만들기
>
> **🎯 배울 것** — 구간을 안 나누고 두 분포의 차이를 재는 법.
>
> **🔬 하는 일** — 정렬된 두 표본을 같이 훑으며 누적 비율의 최대 간격을 찾고, PSI 와 같은 데이터에 돌려 비교합니다.

In [4]:
# 📗 원문의 ks_statistic / ks_pvalue 를 옮깁니다
def KS통계량(표본a, 표본b):
    a, b = sorted(표본a), sorted(표본b)
    n, m = len(a), len(b)
    i = j = 0
    d = 0.0
    while i < n and j < m:
        x = min(a[i], b[j])
        # 같은 값(동점)을 양쪽에서 전부 지나쳐야 올바르게 비교됩니다
        while i < n and a[i] <= x:
            i += 1
        while j < m and b[j] <= x:
            j += 1
        d = max(d, abs(i / n - j / m))
    return d


def KS_p값(d, n, m):
    유효 = math.sqrt(n * m / (n + m))
    람다 = (유효 + 0.12 + 0.11 / 유효) * d
    if 람다 <= 0:
        return 1.0
    합 = 0.0
    for k in range(1, 101):
        합 += (-1) ** (k - 1) * math.exp(-2 * k * k * 람다 * 람다)
    return max(0.0, min(1.0, 2 * 합))


print("같은 데이터에 PSI 와 KS 를 나란히 돌립니다 (기준 2000건, 현재 500건)\n")
print(f"{'현재 창의 분포':<34}{'PSI':>8}{'KS D':>8}{'KS p값':>12}")
print("-" * 64)
for 라벨, 평균, 표준 in [("똑같음 (100, 20)", 100, 20),
                      ("평균만 조금 (104, 20)", 104, 20),
                      ("평균 많이 (118, 24)", 118, 24),
                      ("퍼짐만 늘림 (100, 32)", 100, 32)]:
    r = random.Random(42)
    현재 = [r.gauss(평균, 표준) for _ in range(500)]
    d = KS통계량(기준, 현재)
    print(f"{라벨:<33}{PSI(기준, 현재):>8.3f}{d:>8.3f}{KS_p값(d, 2000, 500):>12.2e}")

print(f"\n=> 맨 아랫줄을 보세요. 평균은 그대로인데 퍼짐만 늘린 경우입니다.")
print(f"   PSI 도 KS 도 둘 다 잡아냅니다. '평균만 보는' 감시보다 나은 점이죠.")
print(f"\n   D 는 0 에서 1 사이라 크기를 가늠하기 쉽습니다. PSI 는 상한이 없고요.")
print(f"   그래서 D 를 '효과 크기' 로 쓰기 좋습니다. 9절에서 그렇게 씁니다.")

같은 데이터에 PSI 와 KS 를 나란히 돌립니다 (기준 2000건, 현재 500건)

현재 창의 분포                               PSI    KS D       KS p값
----------------------------------------------------------------
똑같음 (100, 20)                       0.031   0.046    3.58e-01
평균만 조금 (104, 20)                    0.073   0.127    4.69e-06
평균 많이 (118, 24)                     0.682   0.369    2.50e-48
퍼짐만 늘림 (100, 32)                    0.189   0.114    5.36e-05

=> 맨 아랫줄을 보세요. 평균은 그대로인데 퍼짐만 늘린 경우입니다.
   PSI 도 KS 도 둘 다 잡아냅니다. '평균만 보는' 감시보다 나은 점이죠.

   D 는 0 에서 1 사이라 크기를 가늠하기 쉽습니다. PSI 는 상한이 없고요.
   그래서 D 를 '효과 크기' 로 쓰기 좋습니다. 9절에서 그렇게 씁니다.


> **👀 주목할 곳** — 맨 아랫줄 — 평균은 100 그대로인데 표준편차만 20→32 로 늘린 경우입니다.
>
> **✅ 알게 된 사실** — 평균만 보는 감시는 이걸 놓칩니다. PSI 와 KS 는 **분포 전체**를 보니까 잡아냅니다.
그리고 `D` 는 0~1 범위라 크기를 가늠하기 쉽습니다. PSI 는 상한이 없고요.

> **🧩 이 절의 결론**
>
> KS 통계량 `D` 는 **두 누적분포의 최대 세로 간격**입니다. 구간을 안 나눠도 됩니다.
> 0~1 범위라 **효과 크기**로 쓰기 좋습니다.
>
> **➡️ 다음**
>
> 그런데 p값이 같이 나옵니다. 편해 보이죠.
> ⚠️ **그 p값을 그대로 믿으면 안 됩니다.** 8절입니다.

---

## 8. ⚠️ 표본이 커지면 p값이 거짓말합니다

<sub>📗 원문 "With large samples, the p-value goes to zero for a gap that nobody cares about"</sub>

📗 원문의 주장입니다.

> 표본이 크면 **아무도 신경 안 쓸 간격**에 대해서도 p값이 0 으로 갑니다.
> 창마다 100만 건이면 **0.02 표준편차 이동**도 "유의미" 해집니다.
> `D` 자체를 효과 크기로 쓰고, **`D` 와 p값이 둘 다 기준을 넘을 때만** 경보하세요.

### 🔑 왜 이런 일이 생기나

p값은 **"이 차이가 우연일 확률"** 을 재는데,
표본이 커질수록 우연으로는 설명하기 어려워집니다.

그러니까 p값은 **"차이가 있느냐"** 에 답하지
**"그 차이가 클 만큼 크냐"** 에는 답하지 않습니다.

데이터가 충분히 많으면 **차이는 언제나 있습니다.** 아주 작을 뿐이죠.

> **실험 8** · ⚠️ 신경 안 쓸 차이로 경보 울리기
>
> **🎯 배울 것** — p값 하나로 경보를 걸면 왜 안 되는지 **반례로** 확인합니다.
>
> **🔬 하는 일** — 평균 차이를 **0.02 표준편차**로 고정해 두고 창 크기만 500 → 20만으로 키우며 D 와 p값을 봅니다.

In [5]:
# ✨ 추가 — 차이는 고정, 표본만 키웁니다
print("두 분포의 차이는 0.02 표준편차로 고정입니다. 아무도 신경 안 쓸 크기죠.")
print("창 크기만 키워 봅니다.\n")
print(f"{'창 크기':>10}{'KS D':>10}{'p값':>13}{'p < 0.005?':>13}{'D >= 0.10?':>13}  경보 정책의 판단")
print("-" * 84)
for n in (500, 5_000, 50_000, 200_000):
    r = random.Random(7)
    a = [r.gauss(0, 1) for _ in range(n)]
    b = [r.gauss(0.02, 1) for _ in range(n)]
    d = KS통계량(a, b)
    p = KS_p값(d, n, n)
    p통과 = p < 0.005
    d통과 = d >= 0.10
    판단 = "드리프트!" if (p통과 and d통과) else ("조용 (D 가 작아서 막힘)" if p통과 else "조용")
    print(f"{n:>10,}{d:>10.4f}{p:>13.2e}{'예' if p통과 else '아니오':>12}"
          f"{'예' if d통과 else '아니오':>13}  {판단}")

print(f"\n=> ⚠️ 맨 아랫줄. 창이 20만 건이 되니 p값이 1.4e-08 입니다.")
print(f"   통계적으로는 '확실히 다르다' 는 뜻이에요. 경보가 울릴 상황이죠.")
print(f"\n   그런데 D 는 0.0097 입니다. 두 분포가 사실상 겹쳐 있습니다.")
print(f"   📗 원문의 주장 그대로입니다. 창이 커지면 p값은 0 으로 가는데 D 는 안 갑니다.")
print(f"\n   맨 오른쪽 열을 보세요. D 기준을 같이 걸어 두면 전부 막힙니다.")
print(f"   그래서 '효과 크기도 요구한다'가 9절 경보 정책의 두 번째 규칙입니다.")
print(f"\n   📝 한 줄 요약: p값은 '차이가 있나', D 는 '그 차이가 크냐' 에 답합니다.")
print(f"      경보는 둘 다 '예' 일 때만 울려야 합니다.")

두 분포의 차이는 0.02 표준편차로 고정입니다. 아무도 신경 안 쓸 크기죠.
창 크기만 키워 봅니다.

      창 크기      KS D           p값   p < 0.005?   D >= 0.10?  경보 정책의 판단
------------------------------------------------------------------------------------
       500    0.0600     3.20e-01         아니오          아니오  조용
     5,000    0.0162     5.25e-01         아니오          아니오  조용
    50,000    0.0105     7.84e-03         아니오          아니오  조용
   200,000    0.0097     1.40e-08           예          아니오  조용 (D 가 작아서 막힘)

=> ⚠️ 맨 아랫줄. 창이 20만 건이 되니 p값이 1.4e-08 입니다.
   통계적으로는 '확실히 다르다' 는 뜻이에요. 경보가 울릴 상황이죠.

   그런데 D 는 0.0097 입니다. 두 분포가 사실상 겹쳐 있습니다.
   📗 원문의 주장 그대로입니다. 창이 커지면 p값은 0 으로 가는데 D 는 안 갑니다.

   맨 오른쪽 열을 보세요. D 기준을 같이 걸어 두면 전부 막힙니다.
   그래서 '효과 크기도 요구한다'가 9절 경보 정책의 두 번째 규칙입니다.

   📝 한 줄 요약: p값은 '차이가 있나', D 는 '그 차이가 크냐' 에 답합니다.
      경보는 둘 다 '예' 일 때만 울려야 합니다.


> **👀 주목할 곳** — `D` 열과 `p값` 열이 **반대로 움직입니다.** D 는 0.06 → 0.0097 로 작아지는데 p값은 0.33 → 1.4e-08 로 작아집니다.
>
> **✅ 알게 된 사실** — ⚠️ 창이 20만 건이면 **0.02 표준편차 차이에도** p값이 1.4e-08 입니다.
D 는 0.0097 로 두 분포가 사실상 겹쳐 있는데도요.
**p값은 '차이가 있나', D 는 '그 차이가 크냐'** 에 답합니다. 경보는 둘 다 넘을 때만 울려야 합니다.

> **🧩 이 절의 결론**
>
> ⚠️ p값 하나로는 경보를 걸 수 없습니다. 표본이 크면 **의미 없는 차이에도** 0 으로 갑니다.
> **효과 크기(D)를 함께** 요구해야 합니다.
>
> **➡️ 다음**
>
> 수치형은 PSI 와 KS 로 됩니다. **범주형**은요? 결제 채널 같은 것 말입니다.
> 9절에서 카이제곱을 만들고, 거기서 나오는 효과 크기까지 묶어 경보 정책을 짭니다.

---

## 9. 카이제곱과 경보 정책

<sub>📗 원문 "Chi-square test for categorical features" + "Alert thresholds and false alarms"</sub>

### 범주형 특징은 표로 만듭니다

결제 채널 같은 범주형 특징은 **2행짜리 표**를 만듭니다.
행은 기준·현재, 열은 범주 하나씩이요.

각 칸의 **기대 개수**는 행 합계와 열 합계에서 나옵니다.

$$\text{기대}[r][c] = \frac{\text{행합}[r] \cdot \text{열합}[c]}{\text{전체합}}, \qquad
\chi^2 = \sum_{\text{칸}} \frac{(\text{관측} - \text{기대})^2}{\text{기대}}$$

범주가 `k` 개면 자유도는 `k - 1` 입니다.

> 📗 **현재 창에만 나타난 범주**는 큰 항을 보탭니다.
> 그래서 이 검정은 **새로운 값에 민감하게** 반응합니다. 유용한 성질이죠.

**효과 크기**는 **Cramér's V** 입니다. 2행짜리 표에서는 $\sqrt{\chi^2 / \text{전체합}}$ 이고요.
8절에서 본 `D` 와 같은 역할을 합니다.

---

### 📗 경보가 너무 많으면 아무도 안 봅니다

> 검정마다 아무 일 없을 때의 **거짓 경보율 `alpha`** 가 있습니다.
> 모니터링은 검정을 **많이** 돌립니다.
> 특징 50개를 매시간 `alpha = 0.05` 로 검사하면,
> **아무 일 없어도 하루 약 60번** 거짓 경보가 납니다.
> 사람들은 그 훨씬 전에 알람 읽기를 그만둡니다.

### 📗 경보를 쓸 만하게 만드는 네 규칙

| # | 규칙 | 이 노트북의 설정 |
| --- | --- | --- |
| 1 | **다중 검정 보정.** Bonferroni — `alpha` 를 검정 수로 나눕니다 | 전체 0.01 ÷ 검정 2개 = 각 **0.005** |
| 2 | **효과 크기도 요구.** PSI · D · Cramér's V 도 기준을 넘어야 | D ≥ 0.10, V ≥ 0.10 |
| 3 | **지속성도 요구.** `k` 번 연속 드리프트일 때만 경보 | **k = 2** |
| 4 | **경고와 경보를 나눔.** 경고는 대시보드로, 경보는 사람에게 | 두 등급 |

> 📗 원문이 드는 예: 깨끗한 창 하나가 카이제곱 p값 **0.0009** 를 냅니다.
> 검정별 `alpha` 0.005 보다 작으니 통과죠.
> 그런데 **Cramér's V 가 0.1 미만**이라 정책은 **경보가 아니라 경고**를 남깁니다.

규칙 2가 없었으면 그 창은 사람을 깨웠을 겁니다.

> **실험 9** · 카이제곱 · Cramér's V · 경보 정책
>
> **🎯 배울 것** — 범주형 드리프트를 재고, 네 규칙이 거짓 경보를 **실제로 얼마나** 줄이는지 봅니다.
>
> **🔬 하는 일** — 카이제곱을 만들어 채널 분포 변화에 돌리고, 규칙을 하나씩 켜 가며 경보 수를 셉니다.

In [6]:
# 📗 원문의 chi2_sf / chi_square_test 를 옮깁니다
def _감마하부(s, x, 반복=200):
    # 정칙화 하부 불완전 감마 함수. 급수 전개입니다
    항 = 1.0 / s
    합 = 항
    for k in range(1, 반복):
        항 *= x / (s + k)
        합 += 항
        if abs(항) < abs(합) * 1e-15:
            break
    return 합 * math.exp(-x + s * math.log(x) - math.lgamma(s))


def 카이제곱_생존(통계량, 자유도):
    # P(X > 통계량). 작은 값은 급수, 큰 값은 연분수가 정확합니다
    if 통계량 <= 0:
        return 1.0
    s, x = 자유도 / 2.0, 통계량 / 2.0
    if x < s + 1:
        return max(0.0, min(1.0, 1.0 - _감마하부(s, x)))
    # 연분수 (Lentz 알고리즘)
    작은수 = 1e-300
    b, c, d = x + 1 - s, 1 / 작은수, 1 / (x + 1 - s)
    h = d
    for i in range(1, 300):
        an = -i * (i - s)
        b += 2
        d = an * d + b
        if abs(d) < 작은수:
            d = 작은수
        c = b + an / c
        if abs(c) < 작은수:
            c = 작은수
        d = 1 / d
        변화 = d * c
        h *= 변화
        if abs(변화 - 1) < 1e-15:
            break
    return max(0.0, min(1.0, math.exp(-x + s * math.log(x) - math.lgamma(s)) * h))


def 카이제곱검정(기준개수, 현재개수):
    범주 = [c for c in sorted(set(기준개수) | set(현재개수))
          if 기준개수.get(c, 0) + 현재개수.get(c, 0) > 0]
    행 = [[기준개수.get(c, 0) for c in 범주], [현재개수.get(c, 0) for c in 범주]]
    행합 = [sum(r) for r in 행]
    전체 = sum(행합)
    if len(범주) < 2 or 0 in 행합:
        return 0.0, 0, 1.0, 0.0
    통계량 = 0.0
    for 열 in range(len(범주)):
        열합 = 행[0][열] + 행[1][열]
        for r in range(2):
            기대 = 행합[r] * 열합 / 전체
            통계량 += (행[r][열] - 기대) ** 2 / 기대
    자유도 = len(범주) - 1
    V = math.sqrt(통계량 / 전체)              # 2행 표에서의 Cramér's V
    return 통계량, 자유도, 카이제곱_생존(통계량, 자유도), V


def 채널개수(섞음, n, 시드):
    r = random.Random(시드)
    고름 = r.choices(("app", "store", "web"), weights=섞음, k=n)
    return {c: 고름.count(c) for c in ("app", "store", "web")}


기준채널 = 채널개수((0.35, 0.15, 0.50), 2000, 0)
print("기준 채널 비율: app 35%, store 15%, web 50% (2000건)")
print("현재 창 500건씩 21개를 뽑아 중앙값을 봅니다 (한 창만 보면 운에 휘둘립니다)\n")
print(f"{'현재 창의 채널 비율':<30}{'chi2':>10}{'p값':>11}{'V':>8}  정책 판단")
print("-" * 74)
for 라벨, 섞음 in [("똑같음 (35/15/50)", (0.35, 0.15, 0.50)),
                ("살짝 다름 (38/14/48)", (0.38, 0.14, 0.48)),
                ("많이 다름 (60/10/30)", (0.60, 0.10, 0.30))]:
    모음 = [카이제곱검정(기준채널, 채널개수(섞음, 500, 11 + s)) for s in range(21)]
    통 = sorted(x[0] for x in 모음)[10]
    p = sorted(x[2] for x in 모음)[10]
    V = sorted(x[3] for x in 모음)[10]
    판단 = "드리프트" if (p < 0.005 and V >= 0.10) else ("경고" if p < 0.005 else "정상")
    print(f"{라벨:<29}{통:>10.1f}{p:>11.1e}{V:>8.3f}  {판단}")

print(f"\n=> 가운데 줄을 보세요. 비율이 실제로 조금 다른데도 정상 판정입니다.")
print(f"   500건으로는 3%p 차이를 구별할 수 없거든요. 이게 창 크기의 또 다른 얼굴입니다.")

print("\n\n=== 네 규칙이 거짓 경보를 얼마나 줄이나 ===\n")


@dataclass
class 경보정책:
    검정수: int = 2
    전체알파: float = 0.01
    PSI경고: float = 0.10
    PSI드리프트: float = 0.25
    D드리프트: float = 0.10
    최소V: float = 0.10
    지속: int = 2
    연속: dict = field(default_factory=dict)

    @property
    def 알파(self):
        return self.전체알파 / self.검정수          # 규칙 1: Bonferroni

    def 수준(self, 지표):
        if "psi" in 지표:
            if 지표["psi"] >= self.PSI드리프트 or (지표["ks"] >= self.D드리프트
                                              and 지표["ks_p"] < self.알파):
                return "드리프트"
            return "경고" if 지표["psi"] >= self.PSI경고 else "정상"
        if 지표["p"] < self.알파 and 지표["v"] >= self.최소V:      # 규칙 2
            return "드리프트"
        return "경고" if 지표["p"] < self.알파 else "정상"

    def 갱신(self, 이름, 수준):
        연 = self.연속.get(이름, 0)
        if 수준 == "드리프트":
            연 += 1
            self.연속[이름] = 연
            if 연 == self.지속:                                  # 규칙 3
                return "경보"
        else:
            self.연속[이름] = 0
        return None


특징수, 시간수 = 50, 24
print(f"📗 원문이 드는 상황 그대로입니다. 특징 {특징수}개를 매시간 검사, 하루 {시간수}번.")
print(f"   데이터에 드리프트는 하나도 없습니다.\n")

# 검정 결과를 미리 다 계산해 둡니다 (규칙만 갈아 끼우려고요)
검사 = []
for t in range(시간수):
    for f in range(특징수):
        r = random.Random(5000 + t * 특징수 + f)
        창 = [r.gauss(100, 20) for _ in range(500)]
        d = KS통계량(기준, 창)
        검사.append((PSI(기준, 창), d, KS_p값(d, 2000, 500)))


def 하루경보수(알파, D기준, 지속):
    연속 = {}
    경보 = 0
    for t in range(시간수):
        for f in range(특징수):
            psi, d, p = 검사[t * 특징수 + f]
            if psi >= 0.25 or (d >= D기준 and p < 알파):
                연속[f] = 연속.get(f, 0) + 1
                if 연속[f] == 지속:
                    경보 += 1
            else:
                연속[f] = 0
    return 경보


느슨 = 0.05
보정 = 0.01 / 특징수
print(f"{'켜 둔 규칙':<34}{'하루 경보':>11}")
print("-" * 48)
for 라벨, a, D, k in [("아무 규칙 없음 (p < 0.05 만)", 느슨, 0.0, 1),
                    ("규칙 1 만 — 보정 (alpha 0.01/50)", 보정, 0.0, 1),
                    ("규칙 2 만 — 효과 크기 (D >= 0.10)", 느슨, 0.10, 1),
                    ("규칙 3 만 — 지속성 (연속 2회)", 느슨, 0.0, 2),
                    ("셋 다 켬 (실제 정책)", 보정, 0.10, 2)]:
    print(f"{라벨:<33}{하루경보수(a, D, k):>10}회")

맨위 = 하루경보수(느슨, 0.0, 1)
print(f"\n=> 맨 윗줄: 하루 {맨위}번 울립니다. 드리프트는 0 인데요.")
print(f"   📗 원문은 '약 60번' 이라고 합니다. 산술로는 50 x 24 x 0.05 = 60 이 맞습니다.")
print(f"   측정값이 {맨위}번으로 좀 적은 건 KS p값 근사가 보수적이기 때문입니다.")
print(f"   실질 거짓 양성률이 0.05 가 아니라 {맨위 / (특징수 * 시간수):.3f} 쯤 되는 거죠.")
print(f"   어느 쪽이든 사람이 읽을 수 있는 양이 아닙니다.")
print(f"\n   그리고 가운데 세 줄을 보세요. 규칙 하나만 켜도 거의 다 막힙니다.")
print(f"   셋을 다 쓰는 건 겹겹이 쌓아 두려는 겁니다. 하나가 느슨해도 나머지가 받치도록요.")
print(f"\n   📝 그래서 이건 민감도를 깎는 장치가 아닙니다.")
print(f"      알람을 계속 읽게 만드는 장치입니다. 안 읽히는 알람은 없는 것과 같으니까요.")

기준 채널 비율: app 35%, store 15%, web 50% (2000건)
현재 창 500건씩 21개를 뽑아 중앙값을 봅니다 (한 창만 보면 운에 휘둘립니다)

현재 창의 채널 비율                         chi2         p값       V  정책 판단
--------------------------------------------------------------------------
똑같음 (35/15/50)                      1.6    4.4e-01   0.026  정상
살짝 다름 (38/14/48)                    4.1    1.3e-01   0.040  정상
많이 다름 (60/10/30)                  120.6    6.3e-27   0.220  드리프트

=> 가운데 줄을 보세요. 비율이 실제로 조금 다른데도 정상 판정입니다.
   500건으로는 3%p 차이를 구별할 수 없거든요. 이게 창 크기의 또 다른 얼굴입니다.


=== 네 규칙이 거짓 경보를 얼마나 줄이나 ===

📗 원문이 드는 상황 그대로입니다. 특징 50개를 매시간 검사, 하루 24번.
   데이터에 드리프트는 하나도 없습니다.

켜 둔 규칙                                  하루 경보
------------------------------------------------
아무 규칙 없음 (p < 0.05 만)                    35회
규칙 1 만 — 보정 (alpha 0.01/50)               0회
규칙 2 만 — 효과 크기 (D >= 0.10)                0회
규칙 3 만 — 지속성 (연속 2회)                      1회
셋 다 켬 (실제 정책)                             0회

=> 맨 윗줄: 하루 35번 울립니다. 드리프트는 0 인데요.
   📗 원문은 '약 60번' 이라고 합

> **👀 주목할 곳** — 아래 표 `하루 경보` 열의 맨 윗줄(35회)과 그 아래 세 줄(0~1회).
>
> **✅ 알게 된 사실** — 드리프트가 **0 인 데이터**인데 아무 규칙 없이는 하루 35번 울립니다. 📗 원문의 '약 60번' 과 같은 크기죠.
그런데 **규칙 하나만 켜도 거의 다 막힙니다.** 셋을 다 쓰는 건 겹겹이 쌓아 두려는 겁니다.
민감도를 깎는 장치가 아니라 **알람을 계속 읽게 만드는** 장치입니다.

> **🧩 이 절의 결론**
>
> 범주형은 **카이제곱 + Cramér's V** 로 잽니다. V 가 D 와 같은 효과 크기 역할을 합니다.
> 그리고 **네 규칙**(보정 · 효과 크기 · 지속 · 등급)이 거짓 경보를 걸러 냅니다.
>
> **➡️ 다음**
>
> 도구가 다 모였습니다. **드리프트를 일부러 주입한 흐름**에 전부 돌려 봅니다. 10절입니다.

---

## 10. 드리프트를 주입한 흐름에 전부 돌리기

<sub>📗 원문 "Step 6: A stream with injected drift"</sub>

9000건짜리 결제 이벤트 흐름을 만듭니다.
각 이벤트에는 **금액(`amount`)**, **채널(`channel`)**, **사기 레이블**이 있고요.

📗 원문이 정의한 **네 구간**입니다.

| 이벤트 | 금액 | 채널 비율 (app/store/web) | 레이블 경계 | 무슨 일이 |
| --- | --- | --- | --- | --- |
| 0 ~ 2999 | 평균 100, 표준편차 20 | 35 / 15 / 50 % | 120 | **기준 구간** |
| 3000 ~ 4999 | 평균 118, 표준편차 24 | 60 / 10 / 30 % | 120 | ⬅ **공변량 드리프트** |
| 5000 ~ 6499 | 평균 100, 표준편차 20 | 35 / 15 / 50 % | 120 | 정상으로 복귀 |
| 6500 ~ 8999 | 평균 100, 표준편차 20 | 35 / 15 / 50 % | **105** | ⬅ **개념 드리프트** |

### 🔑 마지막 줄을 잘 보세요

**금액도 채널도 안 변했습니다.** 처음 구간과 완전히 똑같아요.
바뀐 건 **레이블 경계**뿐입니다. 120 → 105.

예전엔 110원짜리 결제가 안전했는데 이제 사기라는 뜻입니다.
**같은 입력, 다른 정답.** 2절에서 본 개념 드리프트죠.

### 설정

- 모델: `amount` 에 대한 임곗값 하나. 처음 2000건으로 맞춥니다
- 창: **500건**씩, **250건**마다 이동 (슬라이딩)
- 레이블 지연: **500건**

> **실험 10** · 전체 파이프라인 실행
>
> **🎯 배울 것** — 어느 감시기가 어느 변화를 잡는지 **한 표에서** 봅니다.
>
> **🔬 하는 일** — 흐름을 만들고 PSI · KS · 카이제곱 · 정확도를 창마다 계산해 경보 정책에 넣습니다.

In [7]:
# 📗 원문의 make_stream / run_monitoring 을 옮깁니다
채널들 = ("app", "store", "web")
구간정의 = [
    (0,    100, 20, (0.35, 0.15, 0.50), 120),
    (3000, 118, 24, (0.60, 0.10, 0.30), 120),
    (5000, 100, 20, (0.35, 0.15, 0.50), 120),
    (6500, 100, 20, (0.35, 0.15, 0.50), 105),
]


def 구간찾기(t):
    현재 = 구간정의[0]
    for s in 구간정의:
        if t >= s[0]:
            현재 = s
    return 현재


def 흐름만들기(n=9000, 시드=7):
    r = random.Random(시드)
    사건들 = []
    for t in range(n):
        _, 평균, 표준, 섞음, 경계 = 구간찾기(t)
        금액 = r.gauss(평균, 표준)
        채널 = r.choices(채널들, weights=섞음)[0]
        레이블 = 1 if 금액 + r.gauss(0, 6) > 경계 else 0
        사건들.append({"t": t, "amount": 금액, "channel": 채널, "label": 레이블})
    return 사건들


def 임곗값맞추기(사건들):
    후보 = sorted(e["amount"] for e in 사건들)
    최적, 최고 = 후보[0], -1.0
    for 임 in 후보[::5]:
        정확 = sum((e["amount"] > 임) == (e["label"] == 1) for e in 사건들) / len(사건들)
        if 정확 > 최고:
            최적, 최고 = 임, 정확
    return 최적


def 감시돌리기(사건들, 기준크기=2000, 창크기=500, 보폭=250, 레이블지연=500, 정확도허용=0.05,
           정책설정=None):
    기준구간 = 사건들[:기준크기]
    임 = 임곗값맞추기(기준구간)
    예측 = [1 if e["amount"] > 임 else 0 for e in 사건들]
    기준정확도 = sum(예측[i] == 사건들[i]["label"] for i in range(기준크기)) / 기준크기

    기준금액 = [e["amount"] for e in 기준구간]
    기준채널수 = {c: sum(1 for e in 기준구간 if e["channel"] == c) for c in 채널들}
    정책 = 경보정책(**(정책설정 or {}))
    창 = deque(maxlen=창크기)
    행들 = []
    for e in 사건들[기준크기:]:
        창.append(e)
        지금 = e["t"] + 1
        if len(창) < 창크기 or (지금 - 기준크기) % 보폭:
            continue
        금액창 = [x["amount"] for x in 창]
        d = KS통계량(기준금액, 금액창)
        지표금액 = {"psi": PSI(기준금액, 금액창), "ks": d,
                 "ks_p": KS_p값(d, len(기준금액), 창크기)}
        현재채널수 = {c: sum(1 for x in 창 if x["channel"] == c) for c in 채널들}
        _, _, p, V = 카이제곱검정(기준채널수, 현재채널수)
        지표채널 = {"p": p, "v": V}

        레이블끝 = 지금 - 레이블지연
        시작 = max(0, 레이블끝 - 창크기)
        정확도 = None
        if 레이블끝 > 시작:
            정확도 = sum(예측[i] == 사건들[i]["label"] for i in range(시작, 레이블끝)) / (레이블끝 - 시작)

        수준 = {"amount": 정책.수준(지표금액), "channel": 정책.수준(지표채널)}
        if 정확도 is not None:
            수준["accuracy"] = "드리프트" if 정확도 < 기준정확도 - 정확도허용 else "정상"
        기록 = [f"{정책.갱신(n, l)} {n}" for n, l in 수준.items() if 정책.갱신 and False]
        기록 = []
        for 이름, l in 수준.items():
            결과 = 정책.갱신(이름, l)
            if 결과:
                기록.append(f"{결과} {이름}")
        행들.append({"끝": 지금, "psi": 지표금액["psi"], "d": d, "chi2p": p,
                    "정확도": 정확도, "수준": 수준, "기록": 기록})
    return 행들, 임, 기준정확도


사건들 = 흐름만들기()
행들, 임곗값, 기준정확도 = 감시돌리기(사건들)

print(f"흐름 9000건. 기준 = 처음 2000건. 창 500, 보폭 250, 레이블 지연 500.")
print(f"주입: 공변량 드리프트 3000~5000, 정상 5000~6500, 개념 드리프트 6500부터.")
print(f"모델: amount > {임곗값:.1f} 이면 사기로 예측. 기준 정확도 {기준정확도:.3f}\n")
print(f"{'끝':>5}{'PSI':>8}{'KS D':>8}{'chi2 p':>10}{'정확도':>9}  금액/채널/정확도{'':<14}기록")
print("-" * 86)
짧게 = {"정상": "정상", "경고": "경고", "드리프트": "드리프트"}
for r in 행들:
    정확 = "  -  " if r["정확도"] is None else f"{r['정확도']:.3f}"
    수준글 = "/".join(짧게[r["수준"].get(k, "정상")] for k in ("amount", "channel", "accuracy")
                   if k in r["수준"])
    print(f"{r['끝']:>5}{r['psi']:>8.3f}{r['d']:>8.3f}{r['chi2p']:>10.1e}{정확:>9}  "
          f"{수준글:<30}{', '.join(r['기록'])}")

흐름 9000건. 기준 = 처음 2000건. 창 500, 보폭 250, 레이블 지연 500.
주입: 공변량 드리프트 3000~5000, 정상 5000~6500, 개념 드리프트 6500부터.
모델: amount > 119.0 이면 사기로 예측. 기준 정확도 0.946

    끝     PSI    KS D    chi2 p      정확도  금액/채널/정확도              기록
--------------------------------------------------------------------------------------
 2500   0.014   0.024   8.4e-01    0.952  정상/정상/정상                      
 2750   0.018   0.032   7.8e-01    0.950  정상/정상/정상                      
 3000   0.010   0.041   3.0e-01    0.932  정상/정상/정상                      
 3250   0.152   0.163   7.6e-06    0.924  드리프트/경고/정상                    
 3500   0.463   0.298   2.8e-26    0.936  드리프트/드리프트/정상                  경보 amount
 3750   0.609   0.343   3.5e-35    0.942  드리프트/드리프트/정상                  경보 channel
 4000   0.708   0.366   4.7e-30    0.926  드리프트/드리프트/정상                  
 4250   0.678   0.349   9.1e-28    0.904  드리프트/드리프트/정상                  
 4500   0.822   0.384   4.5e-35    0.910  드리프트/드리프트/정상                  
 4750   0.883   0.4

📗 원문이 이 표를 **세 토막으로** 읽으라고 합니다.

In [8]:
def 첫경보(행들, 이름, 이후):
    for r in 행들:
        if r["끝"] > 이후 and f"경보 {이름}" in r["기록"]:
            return r["끝"]
    return None


공변량 = 첫경보(행들, "amount", 3000)
개념 = 첫경보(행들, "accuracy", 6500)
이후창 = [r for r in 행들 if r["끝"] > 6500]

print("① 공변량 드리프트 (3000 주입)")
print(f"   금액 경보: {공변량} 번째 이벤트. 지연 {공변량 - 3000}건")
사이 = [r["정확도"] for r in 행들 if 3000 < r["끝"] <= 5000 and r["정확도"]]
print(f"   그동안 정확도: {min(사이):.3f} ~ {max(사이):.3f}")
print(f"   => 입력은 크게 변했는데 정확도는 멀쩡합니다. 레이블 규칙이 안 바뀌었으니까요.")
print(f"      2절의 '공변량 드리프트가 늘 해로운 건 아니다' 가 이겁니다.\n")

print("② 복귀 (5000)")
복귀 = [r["끝"] for r in 행들 if any("해제" in x or "resolved" in x for x in r["기록"])]
깨끗 = [r for r in 행들 if 5500 <= r["끝"] <= 6500]
print(f"   입력이 정상으로 돌아오자 PSI 가 {max(r['psi'] for r in 깨끗):.3f} 이하로 내려갑니다")
print(f"   금액·채널 수준이 전부 '정상': {all(r['수준']['amount'] == '정상' for r in 깨끗)}\n")

print("③ 개념 드리프트 (6500 주입) — 여기가 핵심입니다")
print(f"   이후 창 {len(이후창)}개 중")
print(f"     금액 감시가 울린 창  : {sum(1 for r in 이후창 if r['수준']['amount'] != '정상')}개")
print(f"     채널 감시가 울린 창  : {sum(1 for r in 이후창 if r['수준']['channel'] != '정상')}개")
print(f"     정확도가 드리프트인 창: {sum(1 for r in 이후창 if r['수준'].get('accuracy') == '드리프트')}개")
psi범위 = [r["psi"] for r in 이후창]
print(f"   그 구간 PSI 범위: {min(psi범위):.3f} ~ {max(psi범위):.3f}  (경고선 0.10)")
떨어진 = min(r["정확도"] for r in 이후창 if r["정확도"])
print(f"   정확도: 기준 {기준정확도:.3f} -> {떨어진:.3f}")
print(f"   정확도 경보: {개념} 번째. 지연 {개념 - 6500}건 (그중 500 은 레이블 지연)")

① 공변량 드리프트 (3000 주입)
   금액 경보: 3500 번째 이벤트. 지연 500건
   그동안 정확도: 0.904 ~ 0.942
   => 입력은 크게 변했는데 정확도는 멀쩡합니다. 레이블 규칙이 안 바뀌었으니까요.
      2절의 '공변량 드리프트가 늘 해로운 건 아니다' 가 이겁니다.

② 복귀 (5000)
   입력이 정상으로 돌아오자 PSI 가 0.040 이하로 내려갑니다
   금액·채널 수준이 전부 '정상': True

③ 개념 드리프트 (6500 주입) — 여기가 핵심입니다
   이후 창 10개 중
     금액 감시가 울린 창  : 0개
     채널 감시가 울린 창  : 0개
     정확도가 드리프트인 창: 8개
   그 구간 PSI 범위: 0.015 ~ 0.040  (경고선 0.10)
   정확도: 기준 0.946 -> 0.760
   정확도 경보: 7500 번째. 지연 1000건 (그중 500 은 레이블 지연)


> **👀 주목할 곳** — ③ 토막의 `금액 감시가 울린 창` 과 `채널 감시가 울린 창`. 둘 다 **0개** 입니다.
>
> **✅ 알게 된 사실** — 개념 드리프트 구간 10개 창 전부 입력 감시가 조용합니다. PSI 가 0.015~0.040 으로 경고선 0.10 근처도 못 갑니다.
그런데 정확도는 0.946 에서 0.760 으로 떨어집니다.
**입력만 보고 있었으면 '전부 정상'이라고 보고했을 겁니다.**

> **🧩 이 절의 결론**
>
> 공변량 드리프트는 입력 감시가 500건 만에 잡고, **정확도는 멀쩡합니다.**
> 개념 드리프트는 입력 감시가 **10개 창 전부 못 잡고**, 정확도만 떨어집니다.
> 두 고장의 증상이 정반대입니다.
>
> **➡️ 다음**
>
> 그럼 창을 작게 잡으면 더 빨리 잡을까요?
> ⚠️ 교과서는 '작으면 빠른 대신 시끄럽다'고 합니다. 11절에서 재 봅니다.

---

## 11. ⚠️ 창 크기 — 교과서와 조금 다릅니다

<sub>📗 원문 "A small window reacts quickly but scores noise as drift" + ✨ 측정</sub>

4절에서 본 교과서 설명입니다.

> **작은 창** — 빨리 반응하지만 **잡음을 드리프트로 오인**합니다
> **큰 창** — 안정적이지만 **늦게 보고**합니다

맞교환처럼 들리죠. 그런데 📗 원문의 **연습문제 1번**이 직접 재 보라고 합니다.
재 보면 **맞교환의 한쪽만 나타납니다.**

> **실험 11** · ⚠️ 창 크기를 바꿔 가며 지연과 거짓 경고 재기
>
> **🎯 배울 것** — '작은 창은 시끄럽다'가 **경보 정책이 있을 때도** 맞는 말인지 확인합니다.
>
> **🔬 하는 일** — 📗 연습문제 1번입니다. 창을 200·500·1000·2000 으로 바꿔 ① 깨끗한 구간의 거짓 경고 ② 두 드리프트의 경보 지연을 잽니다.

In [9]:
# 📗 연습문제 1번 — 창 크기만 바꿔 전체를 다시 돌립니다
print(f"{'창 크기':>8}{'깨끗한 구간 거짓 경고':>21}{'공변량 지연':>13}{'개념 지연':>12}")
print("-" * 58)
결과 = {}
for 창 in (200, 500, 1000, 2000):
    rs, _, _ = 감시돌리기(사건들, 창크기=창, 보폭=max(1, 창 // 2))
    # 깨끗한 구간: 2000~3000 과 5500~6500
    거짓 = sum(1 for r in rs
             if ((2000 < r["끝"] <= 3000) or (5500 < r["끝"] <= 6500))
             and any(v != "정상" for k, v in r["수준"].items() if k != "accuracy"))
    a = 첫경보(rs, "amount", 3000)
    c = 첫경보(rs, "accuracy", 6500)
    결과[창] = (거짓, a, c)
    print(f"{창:>8}{거짓:>21}{(a - 3000) if a else 0:>13}{(c - 6500) if c else 0:>12}")

print(f"\n   (공변량 3000 주입, 개념 6500 주입. 레이블 지연은 500 고정)")
print(f"\n=> ⚠️ 두 가지가 눈에 띕니다.")
print(f"\n   ① 지연이 창 크기와 정확히 같습니다.")
for 창, (_, a, c) in 결과.items():
    print(f"      창 {창:>4} -> 공변량 지연 {a - 3000:>4}건  (= 창 크기)"
          f"   개념 지연 {c - 6500:>4}건  (= 창 크기 + 레이블 지연 500)")
print(f"\n      당연합니다. 창이 드리프트 이후 데이터로 완전히 채워져야 판정이 바뀌니까요.")
print(f"      그래서 창 크기를 정하는 건 '얼마나 늦게 알아도 되는가' 를 정하는 일입니다.")
print(f"\n   ② 작은 창이 시끄럽지 않습니다.")
for 창, (거짓, _, _) in 결과.items():
    print(f"      창 {창:>4} -> 깨끗한 구간 거짓 경고 {거짓}건")
print(f"\n      교과서대로면 창 200 이 가장 시끄러워야 하는데 그렇지 않습니다.")
print(f"      9절의 경보 정책이 잡음을 흡수했기 때문입니다.")
print(f"      지속성 규칙(연속 2번)이 어쩌다 한 번 튀는 창을 걸러 내거든요.")

    창 크기         깨끗한 구간 거짓 경고       공변량 지연       개념 지연
----------------------------------------------------------
     200                    0          200         700
     500                    1          500        1000
    1000                    0         1000        1500
    2000                    1         2000        2500

   (공변량 3000 주입, 개념 6500 주입. 레이블 지연은 500 고정)

=> ⚠️ 두 가지가 눈에 띕니다.

   ① 지연이 창 크기와 정확히 같습니다.
      창  200 -> 공변량 지연  200건  (= 창 크기)   개념 지연  700건  (= 창 크기 + 레이블 지연 500)
      창  500 -> 공변량 지연  500건  (= 창 크기)   개념 지연 1000건  (= 창 크기 + 레이블 지연 500)
      창 1000 -> 공변량 지연 1000건  (= 창 크기)   개념 지연 1500건  (= 창 크기 + 레이블 지연 500)
      창 2000 -> 공변량 지연 2000건  (= 창 크기)   개념 지연 2500건  (= 창 크기 + 레이블 지연 500)

      당연합니다. 창이 드리프트 이후 데이터로 완전히 채워져야 판정이 바뀌니까요.
      그래서 창 크기를 정하는 건 '얼마나 늦게 알아도 되는가' 를 정하는 일입니다.

   ② 작은 창이 시끄럽지 않습니다.
      창  200 -> 깨끗한 구간 거짓 경고 0건
      창  500 -> 깨끗한 구간 거짓 경고 1건
      창 1000 -> 깨끗한 구간 거짓 경고 0건
      창 2000 -> 깨끗한 구간 거짓 경고 1건

      교과서대로면 창 2

> **👀 주목할 곳** — `공변량 지연` 열과 `창 크기` 열을 나란히 보세요. **같은 숫자**입니다.
>
> **✅ 알게 된 사실** — ⚠️ 지연은 창 크기와 **정확히 1:1** 입니다. 창 200이면 200건, 창 2000이면 2000건 늦습니다.
그런데 **거짓 경고는 창이 작아도 안 늘었습니다.** 경보 정책이 흡수한 겁니다.
6절에서 창 100의 PSI 가 시끄러웠던 걸 떠올려 보세요. 잡음은 **있었는데 정책이 걸러 낸** 거죠.

### 🔑 그래서 교과서 설명을 이렇게 고쳐 쓸 수 있습니다

| | 📗 교과서 | ✨ 측정 결과 |
| --- | --- | --- |
| 작은 창 | 빠르지만 **시끄럽다** | 빠르고, **경보 정책이 있으면 안 시끄럽다** |
| 큰 창 | 안정적이지만 **늦다** | **정확히 창 크기만큼 늦다** |

**잡음은 분명히 있습니다.** 6절에서 창 100 의 PSI 중앙값이 0.111 이었잖아요.
다만 그 잡음이 **경보까지 도달하지 못할** 뿐입니다.

> 📝 그러니 **정책을 먼저 세우고 창 크기를 정하세요.**
> 정책 없이 PSI 원값만 보면 교과서 설명대로 시끄럽습니다(6절).
> 정책이 있으면 창 크기는 거의 **지연만** 정합니다.

> **🧩 이 절의 결론**
>
> ⚠️ 창 크기는 **지연을 1:1로** 정합니다. 창 500이면 500건 늦게 압니다.
> 그리고 **경보 정책이 있으면 작은 창도 시끄럽지 않습니다.** 잡음은 있지만 정책이 걸러 냅니다.
>
> **➡️ 다음**
>
> 이제 드리프트를 찾았다고 칩시다. **그래서 뭘 해야 하죠?**
> 종류마다 답이 다릅니다. 12절입니다.

---

## 12. 재학습 방아쇠 — 종류별로 할 일

<sub>📗 원문 "Retraining triggers"</sub>

### 📗 재학습을 언제 돌릴지, 방아쇠는 셋입니다

| 방아쇠 | 뜻 | 단점 |
| --- | --- | --- |
| **일정** | 매주/매월 돌림 | 간단하지만 **아무 일 없어도 돌고, 일이 생겨도 기다립니다** |
| **드리프트** | 중요한 특징의 입력 드리프트가 기준을 넘으면 | 개념 드리프트는 못 잡습니다(11절) |
| **성능** | 결과 지표가 기준값 - 허용치 아래로 떨어지면 | 레이블을 기다려야 합니다 |

### 🔑 그런데 "재학습" 이 늘 답은 아닙니다

**드리프트 종류가 할 일을 정합니다.** 📗 원문의 표입니다.

| 드리프트 | 할 일 |
| --- | --- |
| 공변량 드리프트인데 **정확도는 멀쩡** | **보통 아무것도 안 합니다.** 결과 지표를 계속 보세요 |
| 공변량 드리프트가 **데이터가 적던 영역**으로 | 그 영역이 포함된 **최근 데이터로 재학습** |
| 사전확률 드리프트 | **점수를 재보정**하거나 **결정 임곗값을 옮깁니다** |
| 개념 드리프트 | ⚠️ **새 레이블을 먼저 모읍니다.** 옛 레이블로 재학습하면 **옛 개념을 또 배웁니다** |

### 🔑 맨 아랫줄이 가장 중요합니다

개념 드리프트가 났는데 **기존 데이터로 재학습하면 아무것도 안 고쳐집니다.**
$P(y \mid x)$ 가 변한 건데 옛 $y$ 로 다시 배우면 **옛 $P(y \mid x)$ 를 배우는** 거니까요.

**레이블을 다시 모으는 게 먼저**입니다. 그래서 3절의 레이블 지연이 여기서 또 발목을 잡습니다.

### 📗 재학습한 다음에는

1. **가장 최근 레이블 붙은 데이터**로 새 모델을 평가합니다
2. 19단원의 **섀도 → 카나리** 단계로 내보냅니다
3. **기준 창을 새 학습 데이터로 리셋**합니다

3번을 빼먹으면 새 모델이 자기 학습 데이터와 다른 옛 기준과 계속 비교됩니다.

> **실험 12** · 개념 드리프트에 옛 레이블로 재학습하면
>
> **🎯 배울 것** — 📗 원문의 '옛 개념을 또 배운다'가 **진짜 그런지** 확인합니다.
>
> **🔬 하는 일** — 개념 드리프트 이후 구간에서 ① 옛 레이블로 ② 새 레이블로 각각 임곗값을 다시 맞춰 정확도를 비교합니다.

In [10]:
# ✨ 추가 — '옛 레이블로 재학습하면 옛 개념을 또 배운다' 를 확인합니다
드리프트후 = 사건들[6500:]
기준구간 = 사건들[:2000]

# ① 기존(드리프트 전) 데이터로 재학습 = 옛 개념
옛임곗값 = 임곗값맞추기(기준구간)
# ② 드리프트 후 데이터로 재학습 = 새 개념 (새 레이블이 필요합니다)
새임곗값 = 임곗값맞추기(드리프트후)


def 정확도(사건들, 임):
    return sum((e["amount"] > 임) == (e["label"] == 1) for e in 사건들) / len(사건들)


print("개념 드리프트 이후 구간(6500~9000) 에서의 정확도\n")
print(f"{'모델':<40}{'임곗값':>9}{'정확도':>10}")
print("-" * 62)
print(f"{'원래 모델 (처음 2000건으로 학습)':<38}{임곗값:>9.1f}{정확도(드리프트후, 임곗값):>10.3f}")
print(f"{'① 옛 데이터로 재학습 (옛 레이블)':<38}{옛임곗값:>9.1f}{정확도(드리프트후, 옛임곗값):>10.3f}")
print(f"{'② 새 데이터로 재학습 (새 레이블)':<38}{새임곗값:>9.1f}{정확도(드리프트후, 새임곗값):>10.3f}")

print(f"\n=> ① 과 원래 모델이 똑같습니다. 임곗값도 정확도도요.")
print(f"   당연합니다. 같은 데이터로 학습했으니 같은 모델이 나오죠.")
print(f"   재학습 버튼을 눌렀는데 아무것도 안 바뀐 겁니다.")
print(f"\n   ② 는 임곗값이 {옛임곗값:.1f} 에서 {새임곗값:.1f} 로 내려가고")
print(f"   정확도가 {정확도(드리프트후, 옛임곗값):.3f} 에서 {정확도(드리프트후, 새임곗값):.3f} 로 올라갑니다.")
print(f"   레이블 경계가 120 -> 105 로 내려갔으니 임곗값도 따라 내려가야 맞는 거죠.")
print(f"\n   📝 차이는 알고리즘이 아니라 레이블입니다.")
print(f"      개념 드리프트에서 재학습이 통하려면 새 레이블이 먼저 와야 합니다.")
print(f"      그리고 새 레이블은 3절의 레이블 지연만큼 기다려야 옵니다.")

개념 드리프트 이후 구간(6500~9000) 에서의 정확도

모델                                            임곗값       정확도
--------------------------------------------------------------
원래 모델 (처음 2000건으로 학습)                     119.0     0.775
① 옛 데이터로 재학습 (옛 레이블)                      119.0     0.775
② 새 데이터로 재학습 (새 레이블)                      104.6     0.911

=> ① 과 원래 모델이 똑같습니다. 임곗값도 정확도도요.
   당연합니다. 같은 데이터로 학습했으니 같은 모델이 나오죠.
   재학습 버튼을 눌렀는데 아무것도 안 바뀐 겁니다.

   ② 는 임곗값이 119.0 에서 104.6 로 내려가고
   정확도가 0.775 에서 0.911 로 올라갑니다.
   레이블 경계가 120 -> 105 로 내려갔으니 임곗값도 따라 내려가야 맞는 거죠.

   📝 차이는 알고리즘이 아니라 레이블입니다.
      개념 드리프트에서 재학습이 통하려면 새 레이블이 먼저 와야 합니다.
      그리고 새 레이블은 3절의 레이블 지연만큼 기다려야 옵니다.


> **👀 주목할 곳** — `① 옛 데이터로 재학습` 줄과 `원래 모델` 줄. **숫자가 완전히 같습니다.**
>
> **✅ 알게 된 사실** — 재학습을 돌렸는데 모델이 한 글자도 안 바뀝니다. 📗 원문의 '옛 개념을 또 배운다' 가 이겁니다.
새 레이블로 재학습해야 임곗값이 따라 내려가고 정확도가 회복됩니다.
**개념 드리프트의 병목은 알고리즘이 아니라 레이블입니다.**

> **🧩 이 절의 결론**
>
> 드리프트를 찾았다고 바로 재학습하면 안 됩니다. **종류가 할 일을 정합니다.**
> 공변량이면 기다려도 되고, 사전확률이면 임곗값만 옮기고, **개념이면 레이블부터** 모읍니다.
>
> **➡️ 다음**
>
> 13절에서 전체를 정리하고 Ship It 스킬로 마칩니다.
> 그리고 **21단원**에서 또 다른 종류의 불일치를 봅니다. 학습 때와 서빙 때 특징이 달라지는 문제요.

---

## 13. 정리

### ⚠️ 이 노트북에서 원문과 달랐거나, 원문이 한 줄로 넘어간 것

| 📗 원문 | 측정 결과 | 절 |
| --- | --- | --- |
| "PSI 는 구간 수가 늘면 커지고 표본이 커지면 작아진다" (한 문장 경고) | ⚠️ **드리프트 0 인데 밴드를 넘깁니다.** 구간 50개면 PSI 중앙값 0.132(40창 중 38창이 0.1 초과), 창 100건이면 0.111 | 6 |
| "작은 창은 빨리 반응하지만 잡음을 드리프트로 오인한다" | ⚠️ **경보 정책이 있으면 안 시끄럽습니다.** 창 200에서 거짓 경고 0건. 잡음은 있지만 지속성 규칙이 흡수합니다 | 11 |
| 창 크기 맞교환 (정성적) | ✨ **지연 = 창 크기, 정확히 1:1** 입니다. 창 200→200건, 창 2000→2000건 | 11 |

**측정으로 확인된** 원문의 주장들입니다.

| 📗 원문의 주장 | 결과 | 절 |
| --- | --- | --- |
| 드리프트 없는 500건 창의 PSI 는 0.01~0.04 | ✅ 0.009 ~ 0.046 (중앙 0.024) | 5 |
| 큰 표본에선 p값이 쓸모없어진다 | ✅ 20만 건에서 0.02σ 차이에 p = 1.4e-08, D 는 0.0097 | 8 |
| 공변량 드리프트는 입력 감시가 잡고 정확도는 멀쩡 | ✅ 경보 3500, 그동안 정확도 0.904~0.942 | 10 |
| 개념 드리프트는 입력 감시가 못 본다 | ✅ **10개 창 전부 조용.** PSI 0.015~0.040 | 10 |
| 개념 드리프트에 옛 레이블로 재학습하면 소용없다 | ✅ 임곗값·정확도가 **원래 모델과 완전히 동일** | 12 |

### 🔑 한 줄로 꿰면

> **입력 감시는 빠르지만 개념 드리프트에 눈이 멀었고,
> 결과 감시는 정확하지만 레이블 지연만큼 늦습니다.**
> 그래서 둘 다 돌리고, **경보 정책**이 둘을 사람이 읽을 만하게 만듭니다.

그리고 **드리프트를 찾는 것보다 뭘 할지 정하는 게 어렵습니다.**
공변량이면 아무것도 안 하는 게 답일 수 있고, 개념이면 레이블부터 모아야 합니다.

---

## 🔧 실무에서 쓰는 도구들

<sub>📗 원문 "Use It"</sub>

### scipy 와 맞춰 봤다면

> 📗 원문: `ks_statistic` 과 `chi_square_test` 를 scipy 1.18.1 과 대조했습니다.
> **KS 통계량, 카이제곱 통계량, 카이제곱 p값은 부동소수점 정밀도까지 일치**했습니다.
> 무작위 표본 쌍 200개에서 **KS p값만 scipy 대비 최대 18% 차이**가 났고,
> 가장 큰 차이는 **아주 작은 p값**에서 나왔습니다.

차이의 이유도 📗 원문이 설명합니다.

| | 이 노트북 | scipy |
| --- | --- | --- |
| KS p값 | **극한 콜모고로프 분포** | 두 표본이 1만 이하면 **정확법**, 아니면 유효 표본을 반올림한 유한표본 분포 |
| 카이제곱 (범주 2개) | 보정 없음 | 기본으로 **Yates 연속성 보정** |

8절에서 본 대로 **p값 자체를 경보 기준으로 쓰면 안 되니**, 이 정도 차이는 문제가 안 됩니다.

### 📗 모니터링 라이브러리

| 도구 | 기본 동작 |
| --- | --- |
| **Evidently** | 열 종류와 크기로 검정을 고릅니다. 기준이 1000행 이하면 수치형에 KS, 범주형에 카이제곱. 더 크면 **Wasserstein 거리**와 **Jensen-Shannon 발산**(기준 0.1). PSI 도 선택 가능 |
| **NannyML** | **레이블이 오기 전에 성능을 추정**합니다(CBPE). 모델의 예측 확률을 쓰는데 **보정이 잘 돼 있어야** 합니다. 문서가 명시하길 **공변량 이동은 다루지만 개념 드리프트는 못 다룹니다** |
| **TensorFlow Data Validation** | 연속한 데이터 구간을 비교합니다. 범주형에 **L-무한대 거리**, 수치형에 근사 Jensen-Shannon. **학습 데이터와 서빙 데이터 비교**도 하는데, 그게 21단원 주제입니다 |

> 📗 **NannyML 줄을 다시 보세요.** "공변량 이동은 다루지만 개념 드리프트는 못 다룬다."
> 10절의 데모가 **왜 그런지** 보여 줍니다. 개념 드리프트는 **레이블만이 드러냈으니까요.**

밑바닥부터 만들어 본 코드가 **이 도구들이 계산하는 것 전부**입니다.
도구들은 거기에 **저장·스케줄링·대시보드**와 훨씬 많은 검정을 얹은 거고요.

---

## 🚀 Ship It — `skill-drift-monitoring.md`

<sub>📗 원문 저장소의 `20-model-monitoring-and-drift/outputs/skill-drift-monitoring.md` (92줄)</sub>

> 📗 원문: 이 스킬은 **새 모델의 감시 계획을 세우고, 드리프트 경보를 분류**합니다.
> **어느 층이 울렸는지**, **어떤 종류의 드리프트가 증거에 맞는지**,
> 그리고 **기다릴지 · 재보정할지 · 레이블을 모을지 · 재학습할지**를 묻습니다.

### 경보가 왔을 때 묻는 순서

| 순서 | 질문 | 다음 |
| --- | --- | --- |
| 1 | **어느 층**이 울렸나 (시스템/입력/예측/결과) | 입력만이면 2번, 결과면 4번 |
| 2 | **효과 크기**도 기준을 넘었나 (PSI · D · V) | 아니면 경고로 내립니다 |
| 3 | 정확도는 **멀쩡**한가 | 멀쩡하면 공변량 드리프트. 보통 **그대로 둡니다** |
| 4 | 입력은 **조용한데** 정확도만 떨어졌나 | **개념 드리프트.** 새 레이블부터 모읍니다 |
| 5 | 양성 비율만 변했나 | **사전확률 드리프트.** 재보정 또는 임곗값 이동 |

### ✨ 추가 — 측정 뒤에 제가 덧붙인 세 질문

```markdown
## 감시를 처음 세울 때 (⚠️ 6절)

PSI 기준선을 남의 숫자(0.1 / 0.25)로 가져오지 마세요.
**드리프트가 없다고 아는 기간에 돌려 보고** 거기서 나온 최댓값 위로 그으세요.
구간 수와 창 크기를 바꾸면 기준선도 다시 그어야 합니다.

## 창 크기를 정할 때 (⚠️ 11절)

"잡음이 걱정돼서 창을 크게" 는 경보 정책이 있으면 불필요합니다.
창 크기는 사실상 **지연만** 정합니다 (지연 = 창 크기, 1:1).
그러니 물어야 할 질문은 **"얼마나 늦게 알아도 되나"** 입니다.

## 재학습을 결정할 때 (12절)

"재학습하면 되죠?" 에 바로 예라고 하지 마세요.
개념 드리프트에 옛 레이블로 재학습하면 **모델이 한 글자도 안 바뀝니다.**
먼저 물으세요. **"새 레이블이 있습니까?"**
```

---

## 📚 용어 정리

<sub>📗 원문 "Key Terms" — 설명은 쉬운 말로 다시 썼습니다</sub>

| 용어 | 흔히 하는 말 | 진짜 무슨 뜻인가 |
| --- | --- | --- |
| **데이터 드리프트** | "데이터가 변했어요" | 기준 창 대비 입력·예측·레이블 분포가 바뀐 것 |
| **공변량 드리프트** | "새로운 유형의 사용자" | $P(x)$ 는 변하고 $P(y \mid x)$ 는 그대로 |
| **사전확률 드리프트** | "양성이 전보다 많아요" | $P(y)$ 는 변하고 $P(x \mid y)$ 는 그대로. 레이블 시프트라고도 합니다 |
| **개념 드리프트** | "게임의 규칙이 바뀌었어요" | $P(y \mid x)$ 가 변함. **같은 입력의 정답이 달라진 것** |
| **기준 창** | "베이스라인 데이터" | 모든 현재 창이 비교당하는 고정 표본. 보통 학습 데이터 |
| **PSI** | "안정성 점수" | 구간별 (현재비율 − 기준비율) × 로그 비를 더한 대칭 지표 |
| **KS 통계량** | "KS 검정" | 두 누적분포함수 사이의 최대 간격. 0~1 |
| **Cramér's V** | "범주형 효과 크기" | 카이제곱 통계량을 0~1 로 맞춘 값 |
| **레이블 지연** | "정답은 나중에 와요" | 예측 시점과 실제 결과 도착 시점의 간격 |
| **알람 피로** | "호출이 너무 많아요" | 거짓 경보가 대부분이라 사람들이 알람을 무시하게 되는 것 |
| **효과 크기** | — | "차이가 있나"가 아니라 **"그 차이가 크냐"** 에 답하는 값 |

---

## 📖 더 읽을 것

<sub>📗 원문 "Further Reading"</sub>

- [Gama et al., *A Survey on Concept Drift Adaptation* (ACM CSUR 2014)](https://research.tue.nl/en/publications/a-survey-on-concept-drift-adaptation) — 2절의 진짜/가상 드리프트 구분과 시간 양상
- [Rabanser, Günnemann, Lipton, *Failing Loudly* (NeurIPS 2019)](https://arxiv.org/abs/1810.11953) — 시프트 탐지를 위한 두 표본 검정들
- [Lu et al., *Learning under Concept Drift: A Review* (2020)](https://arxiv.org/abs/2004.05785) — 탐지·이해·적응 방법 정리
- [Yurdakul, *Statistical Properties of Population Stability Index* (2018)](https://scholarworks.wmich.edu/dissertations/3208/) — **6절에서 측정한 것의 이론적 배경.** PSI 가 표본 크기와 구간 수에 어떻게 반응하는지
- [scipy.stats.ks_2samp 문서](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ks_2samp.html) — 정확법과 점근법의 차이
- [Evidently, How data drift detection works](https://docs.evidentlyai.com/metrics/explainer_drift) — 열 종류·데이터 크기별 기본 검정 선택
- [NannyML, Performance estimation](https://nannyml.readthedocs.io/en/stable/how_it_works/performance_estimation.html) — CBPE 와 그 전제

---

## ✅ 이해 확인 퀴즈

<sub>📗 원문 저장소의 `20-model-monitoring-and-drift/quiz.json` + ✨ 측정 결과를 묻는 문항</sub>

**1. 입력 지표가 전부 정상인데 정확도만 떨어졌습니다. 무슨 드리프트일까요?**
1. 공변량 드리프트
2. 사전확률 드리프트
3. **개념 드리프트**
4. 드리프트가 아니라 버그입니다

**2. 창마다 20만 건이고 KS p값이 1e-08 입니다. 경보를 울려야 할까요?**
1. 예. p값이 아주 작으니까요
2. **D 값을 봐야 압니다. D 가 작으면 울리면 안 됩니다**
3. 아니오. 표본이 크면 늘 무시합니다
4. 카이제곱으로 다시 검정합니다

**3. ⚠️ PSI 가 0.13 나왔습니다. '중간 정도 이동' 일까요?**
1. 예. 0.1~0.25 구간이니까요
2. **구간 수와 창 크기를 봐야 압니다. 설정에 따라 드리프트 0 에서도 나옵니다**
3. 아니오. 0.25 를 넘어야 의미가 있습니다
4. KS 로 다시 재야 합니다

**4. ⚠️ 창 크기를 500에서 2000으로 키우면?**
1. 거짓 경고가 많이 줄어듭니다
2. **경보가 정확히 1500건 더 늦게 옵니다**
3. 민감도가 올라갑니다
4. PSI 가 커집니다

**5. 개념 드리프트가 났습니다. 바로 재학습을 돌리면?**
1. 성능이 회복됩니다
2. 조금 나아집니다
3. **새 레이블이 없으면 모델이 전혀 안 바뀝니다**
4. 과적합이 생깁니다

In [11]:
문항들 = [
    ("입력은 조용, 정확도만 하락", 3,
     "개념 드리프트는 P(y|x) 가 변한 것이라 입력 분포는 그대로일 수 있습니다. "
     "10절 측정에서 개념 드리프트 이후 10개 창 전부 입력 감시가 조용했고 PSI 는 0.015~0.040 이었습니다. "
     "그 사이 정확도는 0.946 -> 0.760 으로 떨어졌고요.",
     "2절 (세 종류) · 10절 (실제 측정)"),
    ("큰 표본의 작은 p값", 2,
     "20만 건 창에서 0.02 표준편차 차이에 p = 1.4e-08 이 나왔습니다. "
     "그런데 D 는 0.0097 로 두 분포가 사실상 겹쳐 있었습니다. "
     "p값은 '차이가 있나', D 는 '그 차이가 크냐' 에 답합니다.",
     "8절 (표본을 키우며 D 와 p값 비교)"),
    ("PSI 0.146 의 해석", 2,
     "구간을 50개로 쪼개면 드리프트가 0 인 데이터에서도 PSI 중앙값이 0.132 가 나옵니다. "
     "창을 100건으로 줄이면 0.111 이고요. 0.1 / 0.25 밴드는 "
     "구간 10개 + 넉넉한 창을 전제로 한 관행입니다.",
     "6절 (구간 수·창 크기를 바꿔 측정)"),
    ("창 크기를 키우면", 2,
     "측정 결과 지연이 창 크기와 정확히 1:1 이었습니다. 창 200 -> 200건, 창 2000 -> 2000건. "
     "그리고 거짓 경고는 창이 작아도 안 늘었습니다. 경보 정책의 지속성 규칙이 잡음을 흡수하거든요.",
     "11절 (연습문제 1번 측정)"),
    ("개념 드리프트에 즉시 재학습", 3,
     "옛 데이터로 재학습하면 같은 데이터로 학습하는 것이라 임곗값도 정확도도 "
     "원래 모델과 완전히 같았습니다. 새 레이블로 학습해야 임곗값이 따라 내려갑니다. "
     "병목은 알고리즘이 아니라 레이블입니다.",
     "12절 (옛 레이블 vs 새 레이블 재학습)"),
]

print("=" * 80)
for 번호, (제목, 정답, 해설, 어디) in enumerate(문항들, 1):
    print(f"{번호}. {제목}")
    print(f"   정답: {정답}번")
    print(f"   해설: {해설}")
    print(f"   이 노트북: {어디}")
    print("-" * 80)

print("\n채점 기준")
print("   5개 다 맞음 -> 21단원(피처 스토어와 학습-서빙 괴리)으로 넘어가세요")
print("   3~4개 맞음  -> 틀린 문항의 '이 노트북' 절만 다시 읽으세요")
print("   2개 이하    -> 2절(드리프트 세 종류)과 10절(실제 흐름)을 먼저 보세요")

print("\n\n19~21단원이 한 묶음입니다")
print("   19단원  설계    — 배포 전에 정할 것")
print("   20단원  감시    — 배포한 뒤 무엇을 볼 것인가   <- 지금 여기")
print("   21단원  일관성  — 학습 때와 서빙 때 특징이 달라지는 문제")
print("\n   20단원은 '세상이 변하는' 문제였습니다.")
print("   21단원은 '세상은 그대로인데 우리 코드가 두 벌이라서' 생기는 문제입니다.")

1. 입력은 조용, 정확도만 하락
   정답: 3번
   해설: 개념 드리프트는 P(y|x) 가 변한 것이라 입력 분포는 그대로일 수 있습니다. 10절 측정에서 개념 드리프트 이후 10개 창 전부 입력 감시가 조용했고 PSI 는 0.015~0.040 이었습니다. 그 사이 정확도는 0.946 -> 0.760 으로 떨어졌고요.
   이 노트북: 2절 (세 종류) · 10절 (실제 측정)
--------------------------------------------------------------------------------
2. 큰 표본의 작은 p값
   정답: 2번
   해설: 20만 건 창에서 0.02 표준편차 차이에 p = 1.4e-08 이 나왔습니다. 그런데 D 는 0.0097 로 두 분포가 사실상 겹쳐 있었습니다. p값은 '차이가 있나', D 는 '그 차이가 크냐' 에 답합니다.
   이 노트북: 8절 (표본을 키우며 D 와 p값 비교)
--------------------------------------------------------------------------------
3. PSI 0.146 의 해석
   정답: 2번
   해설: 구간을 50개로 쪼개면 드리프트가 0 인 데이터에서도 PSI 중앙값이 0.132 가 나옵니다. 창을 100건으로 줄이면 0.111 이고요. 0.1 / 0.25 밴드는 구간 10개 + 넉넉한 창을 전제로 한 관행입니다.
   이 노트북: 6절 (구간 수·창 크기를 바꿔 측정)
--------------------------------------------------------------------------------
4. 창 크기를 키우면
   정답: 2번
   해설: 측정 결과 지연이 창 크기와 정확히 1:1 이었습니다. 창 200 -> 200건, 창 2000 -> 2000건. 그리고 거짓 경고는 창이 작아도 안 늘었습니다. 경보 정책의 지속성 규칙이 잡음을 흡수하거든요.
   이 노트북: 11절 (연습문제 1